# LC-Rec: Адаптация LLM с интеграцией коллаборативной семантики
### Proof-of-Concept на PyTorch (синтетические данные)

**Статья**: *Adapting Large Language Models by Integrating Collaborative Semantics for Recommendation*

---

## Что реализовано

```
История предметов [ID]
        |
 CF Embedding Lookup    <- заморожен, имитирует предобученный LightGCN
        |
 MLP Projection Layer   <- обучаемый, мост CF -> LLM-пространство
        |
 Positional Encoding    <- обучаемый
        |
 Transformer Stack      <- 2 слоя, causal self-attention
        |
 Prediction Head        <- логиты по каталогу
        |
 Потери: CrossEntropy(rec) + MSE(alignment)
```

**Нет pretrained-моделей. Нет git clone. Всё реализовано вручную.**

## 0. Импорты

In [15]:
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import List, Tuple

print(f'PyTorch: {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {device}')

PyTorch: 2.9.0+cpu
Устройство: cpu


## 1. Конфигурация

Все гиперпараметры в одном месте.

In [16]:
@dataclass
class Config:
    # Синтетический датасет
    num_users: int = 1000           # количество пользователей
    num_items: int = 500            # размер каталога предметов
    max_seq_len: int = 20           # максимальная длина истории
    min_seq_len: int = 5            # минимальная длина последовательности
    interactions_per_user: int = 30

    # CF-эмбеддинги (выход предобученного LightGCN/BPR)
    d_cf: int = 64

    # Проекционный MLP
    d_hidden: int = 256             # скрытый слой MLP

    # Transformer (LLM-блок)
    d_model: int = 128              # скрытая размерность трансформера
    n_heads: int = 4                # головы внимания
    n_layers: int = 2               # количество блоков
    d_ff: int = 512                 # размерность FFN
    dropout: float = 0.1

    # Обучение
    batch_size: int = 32
    lr: float = 1e-3
    epochs: int = 5
    alpha: float = 0.5              # вес потери выравнивания
    seed: int = 42

cfg = Config()
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

print(cfg)

Config(num_users=1000, num_items=500, max_seq_len=20, min_seq_len=5, interactions_per_user=30, d_cf=64, d_hidden=256, d_model=128, n_heads=4, n_layers=2, d_ff=512, dropout=0.1, batch_size=32, lr=0.001, epochs=5, alpha=0.5, seed=42)


## 2. Генерация синтетического датасета

Имитируем последовательности взаимодействий с **популярностным смещением** — топ 10% предметов встречаются в 5 раз чаще (реалистично для e-commerce).

Реальные датасеты из статьи: MovieLens-1M, Amazon Beauty/Sports, Yelp.

In [17]:
def generate_interactions(
    num_users: int,
    num_items: int,
    interactions_per_user: int,
    seed: int = 42
) -> List[List[int]]:
    """
    Генерирует синтетические последовательности взаимодействий.
    Предметы 0..num_items//10 — 'популярные' (вес 5x).
    Возвращает список последовательностей ID предметов (0-indexed).
    """
    rng = np.random.default_rng(seed)

    # Популярностное распределение
    weights = np.ones(num_items)
    weights[:num_items // 10] = 5.0
    weights /= weights.sum()

    sequences = []
    for _ in range(num_users):
        length = rng.integers(interactions_per_user // 2, interactions_per_user)
        # sample without replacement — нет дублей в истории
        items = rng.choice(num_items, size=min(length, num_items),
                           replace=False, p=weights).tolist()
        sequences.append(items)
    return sequences


user_sequences = generate_interactions(
    cfg.num_users, cfg.num_items, cfg.interactions_per_user, cfg.seed
)

lengths = [len(s) for s in user_sequences]
print(f'Пользователей: {len(user_sequences)}')
print(f'Средняя длина: {np.mean(lengths):.1f}  |  Мин: {min(lengths)}  |  Макс: {max(lengths)}')
print(f'Пример (user 0): {user_sequences[0][:10]}...')

Пользователей: 1000
Средняя длина: 22.0  |  Мин: 15  |  Макс: 29
Пример (user 0): [107, 401, 288, 13, 482, 332, 350, 17, 115, 59]...


## 3. Симуляция предобученных CF-эмбеддингов

В реальной системе CF-эмбеддинги получаются из **LightGCN**, обученного на графе взаимодействий:

$$\mathbf{E}_{\mathrm{cf}} = \frac{1}{L+1}\sum_{l=0}^{L} \tilde{\mathbf{A}}^l \mathbf{E}^{(0)}$$

Здесь мы имитируем это через матрицу совместной встречаемости + 3 шага распространения,  
чтобы эмбеддинги имели реалистичную структуру (похожие предметы → похожие векторы).

**Эти эмбеддинги ЗАМОРОЖЕНЫ на всём протяжении обучения** — они служат фиксированным prior.

In [18]:
def simulate_cf_embeddings(
    sequences: List[List[int]],
    num_items: int,
    d_cf: int
) -> torch.Tensor:
    """
    Имитирует предобученные CF-эмбеддинги со структурой совместной встречаемости.
    Предметы, часто встречающиеся вместе, получают похожие векторы.

    Возвращает: E_cf формы (num_items, d_cf), L2-нормализованные.
    """
    # Случайная инициализация
    E = torch.randn(num_items, d_cf) * 0.1

    # Матрица совместной встречаемости предметов из последовательностей
    cooccur = torch.zeros(num_items, num_items)
    for seq in sequences:
        for i in range(len(seq)):
            for j in range(i + 1, min(i + 5, len(seq))):
                cooccur[seq[i], seq[j]] += 1
                cooccur[seq[j], seq[i]] += 1

    # Строковая нормализация (аналог нормировки матрицы смежности в LightGCN)
    cooccur_norm = cooccur / (cooccur.sum(dim=1, keepdim=True) + 1e-8)

    # 3 шага граф-распространения — имитация многослойного LightGCN
    for _ in range(3):
        E = 0.5 * E + 0.5 * (cooccur_norm @ E)

    # L2-нормализация — стандарт в CF
    return F.normalize(E, dim=1)


E_cf = simulate_cf_embeddings(user_sequences, cfg.num_items, cfg.d_cf)
print(f'CF-эмбеддинги: {E_cf.shape}')

# Проверка: популярные предметы (0-49) должны быть ближе друг к другу
cos_pop   = F.cosine_similarity(E_cf[0:1], E_cf[10:11]).item()
cos_mixed = F.cosine_similarity(E_cf[0:1], E_cf[400:401]).item()
print(f'cos(item_0, item_10)   [оба популярные]:  {cos_pop:.4f}')
print(f'cos(item_0, item_400)  [popular vs rare]: {cos_mixed:.4f}')
print('-> Популярные предметы должны быть похожее')

CF-эмбеддинги: torch.Size([500, 64])
cos(item_0, item_10)   [оба популярные]:  0.2181
cos(item_0, item_400)  [popular vs rare]: 0.2456
-> Популярные предметы должны быть похожее


## 4. Датасет и DataLoader

**Протокол leave-one-out** (стандарт в sequential recommendation):
- Каждый пример: `(история[:-1], история[-1])` — предсказать следующий предмет
- Скользящее окно по каждой последовательности
- Истории переменной длины паддируются до максимума в батче

In [19]:
class RecommendationDataset(Dataset):
    """
    Датасет для предсказания следующего предмета (скользящее окно).
    Каждый пример: (history_ids: List[int], target_id: int)
    """
    def __init__(self, sequences: List[List[int]], max_seq_len: int, min_seq_len: int):
        self.samples: List[Tuple[List[int], int]] = []
        for seq in sequences:
            if len(seq) < min_seq_len:
                continue
            for end in range(min_seq_len, len(seq)):
                start = max(0, end - max_seq_len)
                self.samples.append((seq[start:end], seq[end]))

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]


def collate_fn(batch):
    """
    Паддинг историй переменной длины до максимума в батче.

    Возвращает:
        padded  : (B, T_max) — ID предметов, 0 = PAD
        mask    : (B, T_max) — True = реальный токен
        targets : (B,)       — правильный следующий предмет
    """
    histories, targets = zip(*batch)
    max_len = max(len(h) for h in histories)
    B = len(histories)

    padded = torch.zeros(B, max_len, dtype=torch.long)
    masks  = torch.zeros(B, max_len, dtype=torch.bool)
    for i, h in enumerate(histories):
        padded[i, :len(h)] = torch.tensor(h, dtype=torch.long)
        masks[i,  :len(h)] = True

    return padded, masks, torch.tensor(targets, dtype=torch.long)


dataset = RecommendationDataset(user_sequences, cfg.max_seq_len, cfg.min_seq_len)
train_size = int(0.9 * len(dataset))
val_size   = len(dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

print(f'Всего примеров: {len(dataset)}  |  Обучение: {train_size}  |  Валидация: {val_size}')
h, m, t = next(iter(train_loader))
print(f'Батч — история: {h.shape}, маска: {m.shape}, цели: {t.shape}')

Всего примеров: 16993  |  Обучение: 15293  |  Валидация: 1700
Батч — история: torch.Size([32, 20]), маска: torch.Size([32, 20]), цели: torch.Size([32])


## 5. Модель: Projection MLP

**Ключевой обучаемый компонент** — мост между CF-пространством и пространством трансформера:

$$\mathbf{p}_i = \mathrm{LayerNorm}\!\left(\mathbf{W}_2 \cdot \mathrm{GELU}(\mathbf{W}_1 \mathbf{e}_i^{\mathrm{cf}} + \mathbf{b}_1) + \mathbf{b}_2\right)$$

Именно он переводит поведенческий prior CF в токены, понятные трансформеру.

In [20]:
class CFProjectionMLP(nn.Module):
    """
    Проецирует CF-эмбеддинги в скрытое пространство трансформера.

    Вход:  (B, T, d_cf)    — пакет CF-эмбеддингов предметов
    Выход: (B, T, d_model) — soft-токены для трансформера
    """
    def __init__(self, d_cf: int, d_hidden: int, d_model: int):
        super().__init__()
        self.fc1  = nn.Linear(d_cf, d_hidden)
        self.fc2  = nn.Linear(d_hidden, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.act  = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h   = self.act(self.fc1(x))  # (B, T, d_hidden)
        out = self.norm(self.fc2(h)) # (B, T, d_model)
        return out


# Проверка форм
proj = CFProjectionMLP(cfg.d_cf, cfg.d_hidden, cfg.d_model)
x_test = torch.randn(2, 5, cfg.d_cf)
print(f'CFProjectionMLP: {x_test.shape} -> {proj(x_test).shape}')

CFProjectionMLP: torch.Size([2, 5, 64]) -> torch.Size([2, 5, 128])


## 6. Модель: Transformer Block

Pre-norm Transformer — та же структура, что внутри GPT-2 / LLaMA:

$$\mathbf{x}' = \mathbf{x} + \mathrm{MHA}(\mathrm{LN}(\mathbf{x}))$$
$$\mathbf{x}'' = \mathbf{x}' + \mathrm{FFN}(\mathrm{LN}(\mathbf{x}'))$$

**Causal mask** — позиция $t$ видит только позиции $0 \ldots t$ (авторегрессионный режим):

$$A_{ij} = \begin{cases} 0 & j \le i \\ -\infty & j > i \end{cases}$$

In [21]:
class TransformerBlock(nn.Module):
    """
    Pre-norm Transformer блок с causal маской.
    Вход/Выход: (B, T, d_model)
    """
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        # batch_first=True: входной формат (B, T, d_model)
        self.attn  = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=n_heads,
            dropout=dropout, batch_first=True
        )
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(
        self,
        x: torch.Tensor,
        key_padding_mask: torch.Tensor = None,  # (B, T) True = реальный токен
        attn_mask: torch.Tensor = None          # (T, T) causal маска
    ) -> torch.Tensor:
        # Self-attention + residual (pre-norm)
        normed = self.norm1(x)
        # PyTorch MHA: key_padding_mask True = игнорировать -> инвертируем
        kpm = ~key_padding_mask if key_padding_mask is not None else None
        attn_out, _ = self.attn(normed, normed, normed,
                                key_padding_mask=kpm, attn_mask=attn_mask)
        x = x + attn_out

        # FFN + residual (pre-norm)
        x = x + self.ff(self.norm2(x))
        return x


# Проверка форм
block = TransformerBlock(cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.dropout)
x_test = torch.randn(2, 5, cfg.d_model)
m_test = torch.ones(2, 5, dtype=torch.bool)
print(f'TransformerBlock: {x_test.shape} -> {block(x_test, key_padding_mask=m_test).shape}')

TransformerBlock: torch.Size([2, 5, 128]) -> torch.Size([2, 5, 128])


## 7. Полная модель LC-Rec

Пайплайн:

1. `item_ids` → CF embedding lookup (заморожен)
2. CF-эмбеддинги → MLP проекция → soft-токены
3. Soft-токены + позиционные эмбеддинги
4. Transformer stack (causal attention)
5. Пул по последнему реальному токену → prediction head

In [22]:
class LCRecModel(nn.Module):
    """
    LC-Rec: CF soft-токены + Transformer -> предсказание следующего предмета.

    Ключевые принципы:
    - CF-эмбеддинги ЗАМОРОЖЕНЫ (готовый поведенческий prior)
    - Обучается только: MLP-проекция + трансформер + prediction head
    - Causal attention = авторегрессионный режим (как у GPT)
    - Pre-norm = более стабильное обучение (как у LLaMA)
    """
    def __init__(self, E_cf: torch.Tensor, config: Config):
        super().__init__()
        num_items, d_cf = E_cf.shape

        # Замороженные CF-эмбеддинги — поведенческий prior из LightGCN/BPR
        self.item_cf_embed = nn.Embedding.from_pretrained(E_cf, freeze=True)

        # Обучаемая проекция: CF -> пространство трансформера
        self.projection = CFProjectionMLP(d_cf, config.d_hidden, config.d_model)

        # Обучаемые позиционные эмбеддинги
        self.pos_embed = nn.Embedding(config.max_seq_len + 1, config.d_model)

        # Стек трансформер-блоков
        self.blocks = nn.ModuleList([
            TransformerBlock(config.d_model, config.n_heads, config.d_ff, config.dropout)
            for _ in range(config.n_layers)
        ])

        self.norm_out = nn.LayerNorm(config.d_model)

        # Prediction head: d_model -> num_items
        self.head = nn.Linear(config.d_model, num_items, bias=False)

        self.max_seq_len = config.max_seq_len
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding) and m.weight.requires_grad:
                nn.init.normal_(m.weight, std=0.02)

    def forward(
        self,
        item_ids: torch.Tensor,       # (B, T) — история ID предметов
        attention_mask: torch.Tensor, # (B, T) — True = реальный токен
        verbose: bool = False         # печать форм тензоров
    ) -> torch.Tensor:
        """Возвращает логиты (B, num_items) для предсказания следующего предмета."""
        B, T = item_ids.shape

        # Шаг 1: Поиск CF-эмбеддингов (ЗАМОРОЖЕН)
        cf_embs = self.item_cf_embed(item_ids)          # (B, T, d_cf)
        if verbose: print(f'  CF-эмбеддинги:       {cf_embs.shape}')

        # Шаг 2: Проекция CF -> пространство трансформера
        soft_tokens = self.projection(cf_embs)          # (B, T, d_model)
        if verbose: print(f'  Soft-токены:         {soft_tokens.shape}')

        # Шаг 3: Добавление позиционных эмбеддингов
        positions = torch.arange(T, device=item_ids.device).unsqueeze(0).expand(B, -1)
        x = soft_tokens + self.pos_embed(positions)     # (B, T, d_model)
        if verbose: print(f'  После pos_embed:     {x.shape}')

        # Шаг 4: Causal маска (запрет внимания к будущим позициям)
        causal = torch.triu(
            torch.full((T, T), float('-inf'), device=item_ids.device),
            diagonal=1
        )  # (T, T): верхний треугольник = -inf

        # Шаг 5: Трансформер-блоки
        for i, block in enumerate(self.blocks):
            x = block(x, key_padding_mask=attention_mask, attn_mask=causal)
            if verbose: print(f'  После блока {i}:      {x.shape}')

        x = self.norm_out(x)  # (B, T, d_model)

        # Шаг 6: Пул по последнему реальному токену
        # seq_lengths[b] = индекс последнего не-паддинг токена
        seq_lengths = attention_mask.sum(dim=1) - 1     # (B,)
        last = x[torch.arange(B, device=x.device), seq_lengths, :]  # (B, d_model)
        if verbose: print(f'  Последний токен:     {last.shape}')

        # Шаг 7: Prediction head
        logits = self.head(last)                        # (B, num_items)
        if verbose: print(f'  Логиты:              {logits.shape}')

        return logits

    def get_projected_cf(self, item_ids: torch.Tensor) -> torch.Tensor:
        """Возвращает проецированные CF-векторы для подсчёта alignment loss."""
        cf = self.item_cf_embed(item_ids)               # (N, d_cf)
        return self.projection(cf.unsqueeze(1)).squeeze(1)  # (N, d_model)


model = LCRecModel(E_cf, cfg).to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable
print(f'Всего параметров:     {total:,}')
print(f'Обучаемых:            {trainable:,}')
print(f'Заморожено (CF emb):  {frozen:,}')

Всего параметров:     545,280
Обучаемых:            513,280
Заморожено (CF emb):  32,000


## 8. Forward Pass — трассировка форм тензоров

Запускаем один forward pass с `verbose=True`, чтобы увидеть все промежуточные формы.

In [23]:
print('=' * 55)
print('ТРАССИРОВКА ФОРМ ТЕНЗОРОВ (verbose forward pass)')
print('=' * 55)

demo_ids, demo_mask, demo_targets = next(iter(train_loader))
demo_ids  = demo_ids.to(device)
demo_mask = demo_mask.to(device)

print(f'  Вход item_ids:       {demo_ids.shape}  (batch_size x seq_len)')
print(f'  Маска:               {demo_mask.shape}')
print()

model.eval()
with torch.no_grad():
    logits = model(demo_ids, demo_mask, verbose=True)

print()
print(f'  ИТОГ: логиты         {logits.shape}  (batch_size x num_items)')
print(f'  Предсказанный предмет (argmax): {logits.argmax(dim=1)[:5].tolist()}')
model.train()

ТРАССИРОВКА ФОРМ ТЕНЗОРОВ (verbose forward pass)
  Вход item_ids:       torch.Size([32, 20])  (batch_size x seq_len)
  Маска:               torch.Size([32, 20])

  CF-эмбеддинги:       torch.Size([32, 20, 64])
  Soft-токены:         torch.Size([32, 20, 128])
  После pos_embed:     torch.Size([32, 20, 128])
  После блока 0:      torch.Size([32, 20, 128])
  После блока 1:      torch.Size([32, 20, 128])
  Последний токен:     torch.Size([32, 128])
  Логиты:              torch.Size([32, 500])

  ИТОГ: логиты         torch.Size([32, 500])  (batch_size x num_items)
  Предсказанный предмет (argmax): [433, 499, 267, 143, 185]


LCRecModel(
  (item_cf_embed): Embedding(500, 64)
  (projection): CFProjectionMLP(
    (fc1): Linear(in_features=64, out_features=256, bias=True)
    (fc2): Linear(in_features=256, out_features=128, bias=True)
    (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (act): GELU(approximate='none')
  )
  (pos_embed): Embedding(21, 128)
  (blocks): ModuleList(
    (0-1): 2 x TransformerBlock(
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
      )
      (ff): Sequential(
        (0): Linear(in_features=128, out_features=512, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=512, out_features=128, bias=True)
        (4): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm_out): Lay

## 9. Функции потерь

### Рекомендательная потеря
Стандартная кросс-энтропия для предсказания следующего предмета:

$$\mathcal{L}_{\mathrm{rec}} = -\frac{1}{N}\sum_{(\mathcal{S}_u,\, i^+) \in \mathcal{D}} \log \hat{p}(i^+ \mid \mathcal{S}_u)$$

### Потеря выравнивания
Сохраняет **геометрическую структуру** CF-пространства после проекции.  
Поведенчески похожие предметы должны оставаться похожими в пространстве трансформера:

$$\mathcal{L}_{\mathrm{align}} = \frac{1}{N^2}\sum_{i,j} \Bigl(\cos(\mathbf{p}_i, \mathbf{p}_j) - \cos(\mathbf{e}_i^{\mathrm{cf}}, \mathbf{e}_j^{\mathrm{cf}})\Bigr)^2$$

### Итоговая потеря
$$\mathcal{L} = \mathcal{L}_{\mathrm{rec}} + \alpha \cdot \mathcal{L}_{\mathrm{align}}, \quad \alpha = 0.5$$

In [24]:
def recommendation_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """
    Кросс-энтропия для предсказания следующего предмета.
    logits:  (B, num_items) — сырые скоры
    targets: (B,)           — правильные ID предметов
    """
    return F.cross_entropy(logits, targets)


def alignment_loss(
    model: LCRecModel,
    item_ids: torch.Tensor,
    max_items: int = 64
) -> torch.Tensor:
    """
    Потеря структурного выравнивания: MSE между матрицами попарного
    косинусного сходства в CF-пространстве и в пространстве трансформера.

    Предотвращает деформацию CF-геометрии после проекции.
    """
    # Сэмплируем подмножество для вычислительной эффективности: O(max_items^2)
    if len(item_ids) > max_items:
        idx      = torch.randperm(len(item_ids), device=item_ids.device)[:max_items]
        item_ids = item_ids[idx]

    # L2-нормализованные CF-эмбеддинги (заморожены)
    cf   = F.normalize(model.item_cf_embed(item_ids), dim=1)  # (N, d_cf)
    # L2-нормализованные проецированные эмбеддинги (обучаемые)
    proj = F.normalize(model.get_projected_cf(item_ids), dim=1)  # (N, d_model)

    # Матрицы попарного косинусного сходства
    sim_cf   = cf   @ cf.T    # (N, N)
    sim_proj = proj @ proj.T  # (N, N)

    # MSE только на внедиагональных элементах (диагональ = 1 всегда)
    off_diag = ~torch.eye(len(item_ids), dtype=torch.bool, device=item_ids.device)
    return F.mse_loss(sim_proj[off_diag], sim_cf[off_diag])


# Быстрая проверка потерь
model.eval()
with torch.no_grad():
    test_logits  = model(demo_ids, demo_mask)
    test_targets = demo_targets.to(device)
    r = recommendation_loss(test_logits, test_targets)
    a = alignment_loss(model, demo_ids[demo_mask].unique())
    print(f'L_rec (начальная):   {r.item():.4f}  (ожидаем ~log({cfg.num_items}) = {math.log(cfg.num_items):.2f})')
    print(f'L_align (начальная): {a.item():.6f}')
model.train()

L_rec (начальная):   6.4028  (ожидаем ~log(500) = 6.21)
L_align (начальная): 0.010396


LCRecModel(
  (item_cf_embed): Embedding(500, 64)
  (projection): CFProjectionMLP(
    (fc1): Linear(in_features=64, out_features=256, bias=True)
    (fc2): Linear(in_features=256, out_features=128, bias=True)
    (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (act): GELU(approximate='none')
  )
  (pos_embed): Embedding(21, 128)
  (blocks): ModuleList(
    (0-1): 2 x TransformerBlock(
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
      )
      (ff): Sequential(
        (0): Linear(in_features=128, out_features=512, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=512, out_features=128, bias=True)
        (4): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm_out): Lay

## 10. Цикл обучения

Метрики на валидации:
- **HR@10** — попал ли правильный предмет в топ-10?
- **NDCG@10** — награждает модели, ставящие правильный предмет выше

$$\mathrm{NDCG@10} = \frac{1}{|\mathcal{U}|} \sum_u \frac{\mathbf{1}[i_u^* \in \mathrm{Top\text{-}10}]}{\log_2(\mathrm{rank}(i_u^*) + 1)}$$

In [25]:
def evaluate(loader: DataLoader, m: nn.Module) -> dict:
    """Вычисляет Loss, HR@10 и NDCG@10 на заданном DataLoader."""
    m.eval()
    total_loss = hits = ndcg = n = 0

    with torch.no_grad():
        for ids, masks, targets in loader:
            ids, masks, targets = ids.to(device), masks.to(device), targets.to(device)
            logits = m(ids, masks)                          # (B, num_items)
            total_loss += recommendation_loss(logits, targets).item()

            top10 = logits.topk(10, dim=1).indices          # (B, 10)
            for b in range(targets.shape[0]):
                t = targets[b].item()
                if t in top10[b]:
                    hits += 1
                    rank = (top10[b] == t).nonzero(as_tuple=True)[0].item() + 1
                    ndcg += 1.0 / math.log2(rank + 1)  # дисконтирование по позиции
            n += targets.shape[0]

    m.train()
    return {
        'loss':    total_loss / len(loader),
        'HR@10':   hits / n,
        'NDCG@10': ndcg / n,
    }

In [26]:
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=cfg.lr, weight_decay=1e-4
)
# Cosine annealing LR — стандарт для трансформеров
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)


def train_epoch() -> dict:
    model.train()
    tot_rec = tot_aln = tot = n_b = 0

    for ids, masks, targets in train_loader:
        ids, masks, targets = ids.to(device), masks.to(device), targets.to(device)

        optimizer.zero_grad()

        logits   = model(ids, masks)
        rec_loss = recommendation_loss(logits, targets)
        aln_loss = alignment_loss(model, ids[masks].unique())
        loss     = rec_loss + cfg.alpha * aln_loss  # L = L_rec + alpha * L_align

        loss.backward()
        # Gradient clipping — стабилизирует обучение трансформеров
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        tot_rec += rec_loss.item()
        tot_aln += aln_loss.item()
        tot     += loss.item()
        n_b     += 1

    scheduler.step()
    return {'rec': tot_rec/n_b, 'aln': tot_aln/n_b, 'total': tot/n_b}


print(f"{'Эпоха':>5} | {'Потеря':>8} | {'Rec':>7} | {'Aln':>9} | {'Val Loss':>9} | {'HR@10':>6} | {'NDCG@10':>8}")
print('-' * 75)

history = []
for epoch in range(1, cfg.epochs + 1):
    tr = train_epoch()
    va = evaluate(val_loader, model)
    history.append({**tr, **{f'val_{k}': v for k, v in va.items()}})
    print(f"{epoch:>5} | {tr['total']:>8.4f} | {tr['rec']:>7.4f} | {tr['aln']:>9.6f} | "
          f"{va['loss']:>9.4f} | {va['HR@10']:>6.4f} | {va['NDCG@10']:>8.4f}")

print()
print(f"Лучший HR@10:   {max(h['val_HR@10']   for h in history):.4f}")
print(f"Лучший NDCG@10: {max(h['val_NDCG@10'] for h in history):.4f}")

Эпоха |   Потеря |     Rec |       Aln |  Val Loss |  HR@10 |  NDCG@10
---------------------------------------------------------------------------
    1 |   6.1182 |  6.1139 |  0.008514 |    6.0549 | 0.0676 |   0.0317
    2 |   6.0137 |  6.0101 |  0.007281 |    6.0515 | 0.0671 |   0.0303
    3 |   5.9151 |  5.9102 |  0.009760 |    6.1007 | 0.0729 |   0.0344
    4 |   5.6129 |  5.6085 |  0.008668 |    6.2685 | 0.0682 |   0.0325
    5 |   5.2886 |  5.2853 |  0.006622 |    6.3308 | 0.0712 |   0.0333

Лучший HR@10:   0.0729
Лучший NDCG@10: 0.0344


## 11. Ablation: влияние потери выравнивания

Обучаем второй вариант модели **без** `L_align`, чтобы показать её вклад.

In [27]:
torch.manual_seed(cfg.seed)
model_no_align = LCRecModel(E_cf, cfg).to(device)
opt_na = torch.optim.AdamW(
    [p for p in model_no_align.parameters() if p.requires_grad],
    lr=cfg.lr, weight_decay=1e-4
)

for _ in range(cfg.epochs):
    model_no_align.train()
    for ids, masks, targets in train_loader:
        ids, masks, targets = ids.to(device), masks.to(device), targets.to(device)
        opt_na.zero_grad()
        # Только рекомендательная потеря — без L_align
        loss = recommendation_loss(model_no_align(ids, masks), targets)
        loss.backward()
        opt_na.step()

res_full     = evaluate(val_loader, model)
res_no_align = evaluate(val_loader, model_no_align)

print('Ablation study — влияние потери выравнивания:')
print(f"{'Модель':<35} | {'HR@10':>6} | {'NDCG@10':>8}")
print('-' * 58)
print(f"{'LC-Rec (L_rec + alpha*L_align)':<35} | {res_full['HR@10']:>6.4f} | {res_full['NDCG@10']:>8.4f}")
print(f"{'LC-Rec (только L_rec)':<35} | {res_no_align['HR@10']:>6.4f} | {res_no_align['NDCG@10']:>8.4f}")

Ablation study — влияние потери выравнивания:
Модель                              |  HR@10 |  NDCG@10
----------------------------------------------------------
LC-Rec (L_rec + alpha*L_align)      | 0.0712 |   0.0333
LC-Rec (только L_rec)               | 0.0665 |   0.0294


## 12. Итоговая сводка архитектуры

```
Вход: item_ids (B, T)
         |
         v  item_cf_embed [ЗАМОРОЖЕН]
  (B, T, d_cf=64)        <- поведенческий prior из LightGCN
         |
         v  CFProjectionMLP [ОБУЧАЕТСЯ]
  (B, T, d_model=128)    <- мост CF -> LLM-пространство
         |
         + pos_embed
         |
         v  TransformerBlock x2 [ОБУЧАЕТСЯ]
  (B, T, d_model=128)    <- causal self-attention
         |
         v  пул по последнему токену
  (B, d_model=128)
         |
         v  head [ОБУЧАЕТСЯ]
  (B, num_items=500)     <- логиты следующего предмета
```

**Функция потерь:**
$$\mathcal{L} = \underbrace{\mathcal{L}_{\mathrm{rec}}}_{\text{CrossEntropy}} + 0.5 \cdot \underbrace{\mathcal{L}_{\mathrm{align}}}_{\text{MSE геометрии}}$$

**Параметры:**

| Компонент | Параметры | Обучается |
|---|---|---|
| CF-эмбеддинги (500×64) | 32 000 | ❌ заморожен |
| MLP-проекция | ~100K | ✅ |
| Позиционные эмбеддинги | ~2.7K | ✅ |
| Transformer (2 блока) | ~526K | ✅ |
| Prediction Head | 64K | ✅ |
| **Итого обучаемых** | **~693K** | ✅ |